# Statement 5 
## Train a DNN on the Forest Fire dataset using RMSprop optimizer with a learning rate of 0.01. Report 
## training and validation accuracy

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (accuracy_score, precision_score, 
                           recall_score, f1_score, roc_auc_score)

In [ ]:
# 1. Data Loading and EDA
train_dir = 'DL_LAB_EXAM/Datasets/forest_fire/Training and Validation'
test_dir = 'DL_LAB_EXAM/Datasets/forest_fire/Testing'

# Data generators
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary'
)

# EDA
print("Class indices:", train_generator.class_indices)
print("Training samples:", train_generator.samples)
print("Validation samples:", val_generator.samples)
print("Test samples:", test_generator.samples)

# Plot class distribution
plt.figure(figsize=(8, 5))
sns.countplot(x=train_generator.classes)
plt.title('Class Distribution in Training Set')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

In [ ]:
# 2. Model Building and Training - Adam
model_adam = Sequential([
    Flatten(input_shape=(150, 150, 3)),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_adam.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_adam = model_adam.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // 32,
    validation_data=val_generator,
    validation_steps=val_generator.samples // 32,
    epochs=15
)

In [ ]:
# 3. Model Building and Training - SGD
model_sgd = Sequential([
    Flatten(input_shape=(150, 150, 3)),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_sgd.compile(
    optimizer=SGD(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_sgd = model_sgd.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // 32,
    validation_data=val_generator,
    validation_steps=val_generator.samples // 32,
    epochs=15
)

In [ ]:
# 5. Evaluation and Comparison
def evaluate_model(model, generator):
    y_true = generator.classes
    y_pred_probs = model.predict(generator).ravel()
    y_pred = (y_pred_probs > 0.5).astype(int)
    
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC AUC': roc_auc_score(y_true, y_pred_probs)
    }

# Evaluate both models
adam_metrics = evaluate_model(model_adam, test_generator)
sgd_metrics = evaluate_model(model_sgd, test_generator)

# Create metrics DataFrame with numeric values only
metrics_df = pd.DataFrame({
    'Adam': adam_metrics,
    'SGD': sgd_metrics
}).T

# Ensure all values are numeric
metrics_df = metrics_df.apply(pd.to_numeric)

# Metrics Comparison Heatmap (fixed)
plt.figure(figsize=(10, 6))
sns.heatmap(metrics_df, annot=True, cmap='YlGnBu', fmt='.3f')
plt.title('Performance Metrics Heatmap Comparison')
plt.show()

# Additional visualizations
plt.figure(figsize=(15, 5))

# Accuracy comparison
plt.subplot(1, 2, 1)
plt.plot(history_adam.history['accuracy'], label='Adam Train')
plt.plot(history_adam.history['val_accuracy'], label='Adam Val')
plt.plot(history_sgd.history['accuracy'], '--', label='SGD Train')
plt.plot(history_sgd.history['val_accuracy'], '--', label='SGD Val')
plt.title('Accuracy Comparison')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()

# Loss comparison
plt.subplot(1, 2, 2)
plt.plot(history_adam.history['loss'], label='Adam Train')
plt.plot(history_adam.history['val_loss'], label='Adam Val')
plt.plot(history_sgd.history['loss'], '--', label='SGD Train')
plt.plot(history_sgd.history['val_loss'], '--', label='SGD Val')
plt.title('Loss Comparison')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.show()

# Metrics Comparison Bar Plot
plt.figure(figsize=(10, 6))
metrics_df.plot(kind='bar', colormap='viridis')
plt.title('Performance Metrics Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Print final comparison table
print("\nFinal Performance Comparison:")
print(metrics_df)